# TRƯỜNG ĐẠI HỌC CÔNG NGHỆ THÔNG TIN - ĐHQG-HCM
### KHOA KHOA HỌC DỮ LIỆU & TRÍ TUỆ NHÂN TẠO
---
## BÁO CÁO BÀI TẬP THỰC HÀNH: LAB 2.1 - LÀM VIỆC VỚI DỮ LIỆU BỊ THIẾU TRONG THƯ VIỆN PANDAS (MISSING DATA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barackvn/MayHocNangCao/blob/main/Buoi2-KhaiPhaVaXulyDuLieu/Lab2_1_XuLyDuLieuKhuyetThieu_MissingData.ipynb)

* **Môn học:** DS200 - Phân tích dữ liệu lớn (Big Data Analytics)
* **Lớp:** DS200.G11.CN2.TTNT
* **Giảng viên hướng dẫn:** ThS. Hà Minh Tân
* **Học viên / Sinh viên thực hiện:** barackvn
* **Email:** hotro.76@gmail.com
* **Mục tiêu bài lab:** Nhận diện, phân loại và áp dụng các kỹ thuật xử lý dữ liệu khuyết thiếu (Missing Data Imputation) trong Pandas.

## I. CƠ SỞ LÝ THUYẾT VỀ DỮ LIỆU KHUYẾT THIẾU (MISSING DATA)

### 1. Missing Data là gì?
Trong quá trình thu thập và tiền xử lý dữ liệu lớn, **Missing Data** (dữ liệu bị thiếu/khuyết) là hiện tượng các trường thông tin không có giá trị, thường được biểu diễn dưới dạng `NaN` (Not a Number), `NaT` (Not a Time), `None`, hoặc `Null`.

### 2. Nguyên nhân xuất hiện dữ liệu khuyết thiếu:
* Người dùng bỏ sót, không điền thông tin trong biểu mẫu/khảo sát.
* Lỗi truyền nhận, mất mát gói tin trong quá trình truyền tải từ cảm biến/IoT/hệ thống streaming.
* Quá trình hợp nhất (merge/join) các bảng cơ sở dữ liệu không khớp khóa chính/phụ.
* Lỗi chương trình, quá trình trích xuất ETL gặp sự cố.

### 3. Phân loại cơ chế khuyết thiếu:
* **MCAR (Missing Completely at Random):** Xác suất bị thiếu hoàn toàn ngẫu nhiên, không phụ thuộc vào bất kỳ biến quan sát nào khác lẫn giá trị của chính nó.
* **MAR (Missing at Random):** Xác suất bị thiếu phụ thuộc vào các biến quan sát khác trong tập dữ liệu nhưng không phụ thuộc vào chính giá trị bị khuyết.
* **MNAR (Missing Not at Random):** Xác suất bị thiếu phụ thuộc trực tiếp vào chính giá trị của biến đó (ví dụ người có thu nhập quá cao thường từ chối khai báo thu nhập).

### 4. Quy ước của thư viện Pandas:
Pandas xem `None` (của Python) và `np.nan` (của NumPy) là tương đương nhau để thể hiện giá trị khuyết thiếu, đồng thời cung cấp bộ công cụ mạnh mẽ:
* `isnull()` / `isna()`: Phát hiện giá trị khuyết thiếu.
* `notnull()` / `notna()`: Phát hiện giá trị hợp lệ.
* `dropna()`: Loại bỏ các dòng/cột chứa giá trị null.
* `fillna()`: Điền giá trị thay thế (hằng số, mean, median, forward/backward fill).
* `replace()`: Thay thế linh hoạt các giá trị null bằng giá trị đại diện.
* `interpolate()`: Điền giá trị bằng phương pháp nội suy toán học (nội suy tuyến tính...).

---
## II. THỰC HÀNH CHI TIẾT THEO YÊU CẦU BÀI LAB

### Bước 1: Nạp các thư viện cần thiết và thiết lập môi trường

In [ ]:
import pandas as pd
import numpy as np
import os
import requests

print("✅ Đã nạp thành công các thư viện!")
print(f"- Pandas version: {pd.__version__}")
print(f"- NumPy version: {np.__version__}")

### Chuẩn bị dữ liệu thực hành (`employees.csv`)
Để đảm bảo notebook chạy độc lập và trơn tru trên Google Colab mà không phụ thuộc vào việc upload thủ công, đoạn mã sau sẽ tự động tải tập tin `employees.csv` từ kho lưu trữ GitHub của đồ án nếu chưa có sẵn.

In [ ]:
# Kiểm tra và tự động tải tập tin employees.csv nếu chưa có
csv_file = "employees.csv"
if not os.path.exists(csv_file):
    print("Đang tải dữ liệu employees.csv từ kho lưu trữ...")
    url = "https://raw.githubusercontent.com/barackvn/MayHocNangCao/main/Buoi2-KhaiPhaVaXulyDuLieu/employees.csv"
    res = requests.get(url)
    if res.status_code == 200:
        with open(csv_file, "wb") as f:
            f.write(res.content)
        print("✅ Tải thành công employees.csv từ GitHub!")
    else:
        # Dự phòng tải từ nguồn GeeksforGeeks
        backup_url = "https://media.geeksforgeeks.org/wp-content/uploads/employees.csv"
        res_backup = requests.get(backup_url)
        with open(csv_file, "wb") as f:
            f.write(res_backup.content)
        print("✅ Tải thành công employees.csv từ nguồn dự phòng!")
else:
    print(f"✅ Tệp tin '{csv_file}' đã có sẵn trong thư mục làm việc.")

---
### Mục 1: Phát hiện những phần dữ liệu bị thiếu trong DataFrame
* Xây dựng DataFrame mẫu từ từ điển (`dict`) chứa các giá trị `np.nan`.
* Sử dụng hàm `isnull()` để kiểm tra vị trí các giá trị rỗng (trả về ma trận `True`/`False`).

In [ ]:
# Tạo dictionary of lists
data_dict = {
    'Điểm dòng 1': [100, 90, np.nan, 95],
    'Điểm dòng 2': [30, 45, 56, np.nan],
    'Điểm dòng 3': [np.nan, 40, 80, 98]
}

# Tạo DataFrame từ dictionary
df_sample = pd.DataFrame(data_dict)

print("--- DataFrame ban đầu chứa các giá trị NaN: ---")
print(df_sample)

print("\n--- Kết quả phát hiện giá trị rỗng với df.isnull(): ---")
print(df_sample.isnull())

In [ ]:
# Thống kê tổng số lượng giá trị thiếu trên mỗi cột
print("Tổng số giá trị bị thiếu trên mỗi cột:")
print(df_sample.isnull().sum())

---
### Mục 2: Làm việc với tập dữ liệu thực tế (`employees.csv`)
* Nạp tập dữ liệu nhân viên gồm 1000 dòng và 8 thuộc tính.
* Tạo một boolean Series lọc ra tất cả các dòng có cột `Gender` bị khuyết (`NaN`).
* Thống kê và hiển thị danh sách các bản ghi bị khuyết thiếu.

In [ ]:
# Đọc dữ liệu từ file employees.csv
data = pd.read_csv("employees.csv")

print("--- 5 dòng đầu tiên của tập dữ liệu employees.csv: ---")
print(data.head())
print(f"\nKích thước tập dữ liệu: {data.shape[0]} dòng, {data.shape[1]} cột.")

In [ ]:
# Tạo bool series trả về True tại các vị trí có giá trị NaN trong cột 'Gender'
bool_series = pd.isnull(data["Gender"])

# Lọc và hiển thị các bản ghi có cột Gender là NaN
missing_gender_df = data[bool_series]

print(f"Tổng số nhân viên bị thiếu thông tin giới tính: {len(missing_gender_df)} bản ghi")
print("\n--- Danh sách nhân viên có Gender = NaN (trích đoạn): ---")
print(missing_gender_df[["First Name", "Gender", "Senior Management", "Team"]].head(10))

In [ ]:
# Thống kê tỷ lệ khuyết thiếu trên toàn bộ các cột trong dataset
missing_summary = pd.DataFrame({
    'Số lượng khuyết': data.isnull().sum(),
    'Tỷ lệ khuyết (%)': (data.isnull().sum() / len(data)) * 100
})
print("--- Thống kê tình trạng khuyết thiếu của tập dữ liệu employees: ---")
print(missing_summary)

---
### Mục 3: Điền các giá trị còn thiếu bằng `fillna()`, `replace()`, và `interpolate()`

#### 3.1. Phương pháp 1: Sử dụng hàm `fillna()`
##### a. Điền giá trị khuyết bằng một hằng số cụ thể (ví dụ: `0`)
* `df.fillna(0)` sẽ thay thế toàn bộ các giá trị `NaN` trong bảng bằng `0.0`.

In [ ]:
# Tạo lại DataFrame mẫu
data_dict = {
    'Điểm dòng 1': [100, 90, np.nan, 95],
    'Điểm dòng 2': [30, 45, 56, np.nan],
    'Điểm dòng 3': [np.nan, 40, 80, 98]
}
df = pd.DataFrame(data_dict)

print("--- DataFrame ban đầu: ---")
print(df)

# Tìm và thay thế các giá trị còn thiếu bằng 0
df_filled_zero = df.fillna(0)
print("\n--- Kết quả sau khi dùng df.fillna(0): ---")
print(df_filled_zero)

##### b. Điền giá trị còn thiếu bằng giá trị ngay trước đó (`method='pad'` hay forward fill)
* Khi sử dụng `method='pad'`, giá trị `NaN` sẽ được kế thừa giá trị hợp lệ nằm liền kề phía trước của cùng cột đó.
* **Hiện tượng quan sát:** Ở hàng index `0` của cột `Điểm dòng 3`, vì phía trên nó không có giá trị nào trước đó nên giá trị này vẫn giữ nguyên là `NaN`.

In [ ]:
# Điền giá trị còn thiếu với những giá trị trước đó (Forward Fill)
df_filled_pad = df.fillna(method='pad')

print("--- Kết quả sau khi dùng df.fillna(method='pad'): ---")
print(df_filled_pad)

---
#### 3.2. Phương pháp 2: Sử dụng hàm `replace()`
* Hàm `replace(to_replace=np.nan, value=...)` cho phép chỉ định chính xác giá trị cần thay thế.
* Thường dùng để gán một giá trị quy ước thể hiện dữ liệu thiếu (ví dụ `-88` hoặc `-99`) phục vụ cho các thuật toán học máy hoặc chuẩn hóa dữ liệu.

In [ ]:
# Nạp lại dữ liệu employees
data = pd.read_csv("employees.csv")

# Thay thế toàn bộ NaN trong dataframe với giá trị quy ước -88
data_new = data.replace(to_replace=np.nan, value=-88)

print("--- Kết quả sau khi dùng replace(to_replace=np.nan, value=-88): ---")
print(data_new[["First Name", "Gender", "Senior Management", "Team"]].head(10))

# Kiểm tra các vị trí trước đây là NaN đã được chuyển thành -88
print("\nKiểm tra bản ghi dòng 1 (trước đây có Team = NaN):")
print(data_new.loc[1, ["First Name", "Team"]])

---
#### 3.3. Phương pháp 3: Sử dụng hàm `interpolate()` với phương pháp tuyến tính (Linear Interpolation)
* **Nguyên lý:** Nội suy tuyến tính giả định rằng các giá trị biến thiên theo đường thẳng nối giữa hai điểm dữ liệu đã biết trước và sau giá trị khuyết.
* Công thức tính giá trị nội suy:
$$y = y_1 + \frac{x - x_1}{x_2 - x_1} \times (y_2 - y_1)$$
* **Ví dụ minh họa:**
  * Tại cột `C`: giữa hàng 1 (giá trị = 16) và hàng 3 (giá trị = 3), hàng 2 bị khuyết sẽ được nội suy là:
    $$\frac{16 + 3}{2} = 9.5$$
  * Tại cột `D`: giữa hàng 1 (giá trị = 3) và hàng 4 (giá trị = 6), hai vị trí khuyết liên tiếp tại hàng 2 và 3 sẽ được nội suy tuyến tính đều thành `4.0` và `5.0`.

In [ ]:
# Khởi tạo DataFrame mẫu theo đúng đề bài Lab
df_inter = pd.DataFrame({
    "A": [12, 4, 5, None, 1],
    "B": [None, 2, 54, 3, None],
    "C": [20, 16, None, 3, 8],
    "D": [14, 3, None, None, 6]
})

print("--- DataFrame ban đầu trước khi nội suy: ---")
print(df_inter)
print("=" * 45)

# Áp dụng hàm interpolate với phương pháp linear (tuyến tính)
data_new = df_inter.interpolate(method='linear', limit_direction='forward')

print("--- DataFrame sau khi thực hiện interpolate(method='linear'): ---")
print(data_new)

---
## III. BẢNG SO SÁNH VÀ ĐÁNH GIÁ CÁC KỸ THUẬT XỬ LÝ DỮ LIỆU KHUYẾT THIẾU

| Kỹ thuật | Hàm trong Pandas | Ưu điểm | Nhược điểm | Trường hợp nên áp dụng |
| :--- | :--- | :--- | :--- | :--- |
| **Loại bỏ (Deletion)** | `dropna()` | Đơn giản, không sinh ra dữ liệu giả mạo. | Làm mất dữ liệu, giảm kích thước mẫu, dễ gây chệch nếu không phải MCAR. | Khi tỷ lệ khuyết rất nhỏ (< 3%) hoặc tập dữ liệu cực lớn. |
| **Điền hằng số / Đại diện** | `fillna(0)`, `replace()` | Dễ thực hiện, giữ nguyên số lượng mẫu. | Có thể làm sai lệch phân phối dữ liệu gốc. | Dữ liệu dạng danh mục hoặc các biến đếm có ý nghĩa 0 khi không xuất hiện. |
| **Kế thừa giá trị (LOCF / NOCB)** | `fillna(method='pad')` hoặc `bfill()` | Rất tự nhiên đối với dữ liệu chuỗi thời gian (Time-series). | Không áp dụng được nếu vị trí đầu tiên bị rỗng. | Dữ liệu cảm biến, giá chứng khoán, nhiệt độ đo theo chu kỳ. |
| **Nội suy tuyến tính (Interpolation)** | `interpolate(method='linear')` | Duy trì được xu hướng biến thiên liên tục của dữ liệu. | Đòi hỏi dữ liệu có tính chất thứ tự (thời gian, không gian). | Dữ liệu tín hiệu, đo đạc vật lý, chuỗi chỉ số kinh tế liên tục. |
| **Điền giá trị thống kê (Imputation)** | `fillna(df.mean())` / `median()` | Bảo toàn kỳ vọng toán học của biến số. | Làm giảm phương sai, làm nhọn phân phối xác suất. | Biến định lượng phân phối chuẩn (mean) hoặc phân phối lệch (median). |

---
## IV. KẾT LUẬN VÀ BÀI HỌC KINH NGHIỆM
1. **Tầm quan trọng của tiền xử lý:** Trong phân tích dữ liệu lớn (Big Data Analytics), chất lượng dữ liệu đầu vào quyết định trực tiếp đến độ tin cậy của mô hình ("Garbage In, Garbage Out"). Việc phát hiện và xử lý dữ liệu khuyết thiếu là bước bắt buộc đầu tiên.
2. **Lựa chọn phương pháp phù hợp:** Không có một phương pháp nào tối ưu cho mọi bài toán. Cần căn cứ vào bản chất thuộc tính (số học hay phân loại), cơ chế khuyết thiếu (MCAR, MAR, MNAR) và bài toán cụ thể để chọn kỹ thuật điền phù hợp.
3. **Thư viện Pandas:** Cung cấp hệ sinh thái hàm (`isnull`, `fillna`, `replace`, `interpolate`) vô cùng linh hoạt và tối ưu hóa tính toán trên cấu trúc vector/C-backend, giúp xử lý các tập dữ liệu lớn với tốc độ cao.